# VIscaNEr Stage 2C — adaptation on 211 reviewed manual photos

Starts from the completed Stage-2B `joint_best.pt`. All 211 manually boxed photos are retained for audit; only the 99 rows with reviewed catalog identities participate in supervised fine-tuning. The original hard validation split is retained as a forgetting guardrail.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/VIscaNEr_manual_stage2c')
OUTPUT = Path('/kaggle/working/manual_stage2c')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'

def dataset_root(slug, marker):
    roots = set()
    for match in INPUT.rglob(marker):
        for parent in (match, *match.parents):
            if parent.name == slug:
                roots.add(parent); break
    if len(roots) != 1:
        raise RuntimeError(f'Attach exactly one dataset {slug} containing {marker}; found: {sorted(roots)}')
    return roots.pop()

def unique_file(filename, parent_token):
    matches = [p for p in INPUT.rglob(filename) if parent_token in p.as_posix()]
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename} from {parent_token}; found: {matches}')
    return matches[0]

print('SETUP | resolving datasets and checkpoints...', flush=True)
code = dataset_root('viscaner-fusion-stage2-code', 'code_sha256.json')
manual = dataset_root('viscaner-manual-211', 'build_summary.json')
hard = dataset_root('viscaner-fusion-hardset-v1', 'manifest.csv')
labels = dataset_root('viscaner-dinov3-data', 'crops_metadata.csv')
bottles = dataset_root('viscaner-bottle-classifier-data', 'audit_metadata.csv')
label_checkpoint = unique_file('best.pt', 'dinov3_vitb16_labels_hard_finetune')
bottle_checkpoint = unique_file('best.pt', 'dinov3_vitb16_bottles_hard_finetune')
joint_checkpoint = unique_file('joint_best.pt', 'fusion_stage2b')

shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(code, PROJECT, ignore=shutil.ignore_patterns('dataset-metadata.json'))
for archive in PROJECT.glob('*.zip'):
    destination = PROJECT / archive.stem
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(destination)
checksums = json.loads((PROJECT/'code_sha256.json').read_text())
for name, expected in checksums.items():
    actual = hashlib.sha256((PROJECT/name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Code checksum mismatch: {name}')
print(f'SETUP | verified {len(checksums)} versioned code files', flush=True)
sys.path.insert(0, str(PROJECT))
from fusion_stage2.kaggle_io import materialize_dataset_dirs
cache = Path('/kaggle/working/dataset_cache')
manual = materialize_dataset_dirs(manual, ('originals', 'label_crops', 'bottle_crops'), cache)
hard = materialize_dataset_dirs(hard, ('label_crops', 'bottle_crops'), cache)
labels = materialize_dataset_dirs(labels, ('refs', 'crops'), cache)
bottles = materialize_dataset_dirs(bottles, ('refs', 'crops'), cache)
required = (manual/'manifest.csv', manual/'build_summary.json', hard/'manifest.csv', hard/'label_crops', hard/'bottle_crops', labels/'refs', labels/'crops', bottles/'refs', bottles/'crops', label_checkpoint, bottle_checkpoint, joint_checkpoint)
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
manual_summary = json.loads((manual/'build_summary.json').read_text())
if int(manual_summary.get('rows', 0)) != 211 or int(manual_summary.get('trainable_catalog_rows', 0)) != 99:
    raise RuntimeError(f'Unexpected Manual-211 dataset summary: {manual_summary}')
print('SETUP | 211 audit rows; 99 reviewed catalog rows; Stage-2B checkpoint resolved', flush=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=5.6,<6', 'safetensors>=0.5,<1', 'rapidfuzz>=3,<4', 'Unidecode>=1.3,<2'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU accelerator before running this notebook')
print('SETUP | GPU:', torch.cuda.get_device_name(0), '| Manual Stage 2C', flush=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', '-m', 'fusion_stage2.train_manual_211_adaptation',
    '--manual-root', str(manual),
    '--hard-manifest', str(hard/'manifest.csv'), '--hard-root', str(hard),
    '--label-data-root', str(labels), '--bottle-data-root', str(bottles),
    '--label-refs-root', str(labels/'refs'), '--bottle-refs-root', str(bottles/'refs'),
    '--label-base-checkpoint', str(label_checkpoint), '--bottle-base-checkpoint', str(bottle_checkpoint),
    '--joint-checkpoint', str(joint_checkpoint), '--output-dir', str(OUTPUT),
    '--device', 'cuda', '--epochs', '20', '--min-epochs', '5', '--patience', '5',
    '--batch-size', '12', '--eval-batch-size', '48', '--top-k', '10',
    '--unfreeze-last-blocks', '2', '--fusion-lr', '3e-5', '--head-lr', '3e-6',
    '--backbone-lr', '1e-7', '--branch-loss-weight', '0.25',
    '--maximum-hard-val-drop', '0.02', '--amp',
]
print('TRAINING | Stage 2C: continue from joint_best.pt on reviewed manual catalog rows', flush=True)
subprocess.run(command, cwd=PROJECT, check=True)
summary = json.loads((OUTPUT/'final_metrics.json').read_text())
print('FINAL MANUAL STAGE 2C |', json.dumps(summary, ensure_ascii=False), flush=True)
